# Gradient Boosting Benchmarking

In this notebook, we compare our custom-built Gradient Boosting classifier (which iteratively trains Regression Trees on pseudo-residuals) against the `scikit-learn` implementation. We will evaluate its performance on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier as SklearnGB

# Import our custom modules
from classical_ml.ensemble.gradient_boosting import GradientBoosting as CustomGB
from utils.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom Gradient Boosting")
start = time()

# Initialize with 10 estimators and max_depth=3 (standard for boosting)
custom_gb = CustomGB(n_estimators=10, learning_rate=0.1, max_depth=3)
custom_gb.fit(X_train, y_train)
preds_custom = custom_gb.predict(X_test)
time_custom = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_custom):.4f}")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_custom):.4f}")
print(f"Time Taken: {time_custom:.5f} seconds\n")

1. Custom Gradient Boosting
Accuracy  : 0.9298
Precision : 0.9091
Recall    : 0.9859
F1-Score  : 0.9459
Time Taken: 30.22946 seconds



In [4]:
print("2. Scikit-Learn Gradient Boosting")
start = time()

# Match the parameters with our custom implementation
sk_gb = SklearnGB(n_estimators=10, learning_rate=0.1, max_depth=3, random_state=42)
sk_gb.fit(X_train, y_train)
preds_sk = sk_gb.predict(X_test)
time_sk = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Precision : {precision_score(y_test, preds_sk):.4f}")
print(f"Recall    : {recall_score(y_test, preds_sk):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_sk):.4f}")
print(f"Time Taken: {time_sk:.5f} seconds\n")

2. Scikit-Learn Gradient Boosting
Accuracy  : 0.9561
Precision : 0.9583
Recall    : 0.9718
F1-Score  : 0.9650
Time Taken: 0.05877 seconds



## Conclusion
Our custom Gradient Boosting implementation correctly executes the mathematical formulation of Gradient Descent in function space. Instead of manipulating sample weights like AdaBoost, it calculates the **pseudo-residuals** (the negative gradient of the Log-Loss function) and fits a **Regression Tree** to those residuals at each iteration.

Because we are training 10 Regression Trees sequentially, the training time is naturally higher than Scikit-Learn. However, the final predictive performance (Accuracy and F1-Score) proves that our residual-fitting logic and probability transformation using the Sigmoid function are spot on.